# Serving & Inference Architecture

The serving layer is where ML meets SWE — latency budgets, fallback strategies, and retrieval funnels are the core of every production ML system. This note covers the retrieval funnel, ANN retrieval, batch vs online prediction, caching, and fallback design.

## What Interviewers Test
- The candidate generation → ranking → re-ranking funnel with real numbers
- ANN retrieval concepts: brute-force vs LSH vs HNSW
- Batch vs online vs streaming prediction tradeoffs
- Caching strategies for ML results
- Model-in-service vs model-server architectures
- Fallback and degradation design

## The Retrieval Funnel

Every large-scale recommendation/search system uses a funnel to trade precision for latency:

```
10M+ items (full catalog)
  ↓  [ANN retrieval — two-tower dot product]  <10ms
1,000 candidates
  ↓  [Light ranker — logistic/GBDT on features]  <20ms
100 items
  ↓  [Full DNN ranker — all features]  <50ms
10–20 items
  ↓  [Business rules, diversity, dedup]  <5ms
Final result
```

**Why funnel?** The full DNN ranker is too expensive for 10M items; ANN retrieval is too coarse for final ranking. Each stage trades cost for precision.

> 💡 **Interview Tip:** Give numbers — interviewers specifically note whether you have a sense of real latency budgets (10ms, 50ms, 200ms p99) and real sizes (millions to tens).


In [ ]:
import numpy as np
import time
np.random.seed(42)

# --- Brute-force ANN vs approximate ---
n_items, d, n_queries = 50_000, 64, 100
items  = np.random.randn(n_items, d)
items  /= np.linalg.norm(items, axis=1, keepdims=True)
queries = np.random.randn(n_queries, d)
queries /= np.linalg.norm(queries, axis=1, keepdims=True)

# Brute-force: exact top-100
t0 = time.time()
scores_bf = queries @ items.T                          # (n_queries, n_items)
top100_bf  = np.argpartition(-scores_bf, 100, axis=1)[:, :100]
t_bf = time.time() - t0

# Simple LSH approximation
def lsh_retrieve(queries, items, n_hash=16, n_candidates=500):
    """Random projection LSH — approximate retrieval."""
    # Random projection planes
    planes = np.random.randn(n_hash, d)
    item_hashes  = (items  @ planes.T > 0).astype(int)   # (n_items, n_hash)
    query_hashes = (queries @ planes.T > 0).astype(int)  # (n_q, n_hash)
    results = []
    for qh in query_hashes:
        hamming = (item_hashes != qh).sum(axis=1)        # (n_items,)
        cands   = np.argpartition(hamming, n_candidates)[:n_candidates]
        results.append(cands)
    return results

t0 = time.time()
top_lsh = lsh_retrieve(queries, items, n_hash=16, n_candidates=500)
t_lsh = time.time() - t0

# Recall@100
recalls = []
for i in range(n_queries):
    true_set = set(top100_bf[i])
    lsh_set  = set(top_lsh[i])
    top100_lsh = set(np.argpartition(-queries[i] @ items[lsh_set if lsh_set else [0]].T, min(100, len(lsh_set)))[:100])
    recalls.append(len(true_set & top100_lsh) / 100)

print(f"Brute force (exact): {t_bf*1000:.1f}ms for {n_queries} queries")
print(f"LSH (approximate):   {t_lsh*1000:.1f}ms for {n_queries} queries")
print(f"Mean recall@100 of LSH vs brute-force: {np.mean(recalls):.3f}")


## Prediction Modes

| Mode | Latency | Freshness | Use when |
|---|---|---|---|
| **Batch** | Hours | Stale | Low-frequency, expensive models (email recs) |
| **Online (synchronous)** | <200ms p99 | Real-time | Session-based recs, search |
| **Near-real-time (streaming)** | Seconds | Near-fresh | Fraud detection, feed ranking with fresh signals |
| **Async/pre-computed** | <5ms | Moderate | Static item embeddings, user profiles |

**Caching strategies:**
- Cache final ranked result by (user_id, context) with short TTL
- Cache item embeddings (days) — change slowly
- Cache user embeddings (minutes) — change with sessions
- Never cache fraud scores — too staleness-sensitive


## Model-in-Service vs Model-Server

| Architecture | Description | Pros | Cons |
|---|---|---|---|
| **Model-in-service** | Model loaded in the application process | Low latency (no RPC) | Memory in every instance; hard to update |
| **Model-server** | Separate serving process (TorchServe, TF Serving) | Independent scaling; versioned rollouts | Network hop; more infra |
| **Feature + model server** | Separate feature fetch + model inference | Decoupled scaling | Multiple network hops |

> 💡 **Interview Tip:** For real-time ranking at FAANG scale, model-server is standard (TorchServe, custom gRPC). For edge/mobile, model-in-service. The tradeoff is latency vs operational complexity.


## Fallback & Degradation Design

Every serving stack needs graceful degradation:

```
Primary: Full DNN ranker (fresh features)
  ↓ [if latency > SLA]
Secondary: GBDT ranker (cached features)
  ↓ [if model unavailable]
Tertiary: Popularity ranker (pre-computed)
  ↓ [if all fails]
Static: Hand-curated editorial list
```

**Circuit breakers:** Wrap model calls in a circuit breaker (fail-fast if error rate > threshold). Log all fallback events for monitoring.


## Common Interview Questions

**Q: Why do large-scale recommendation systems use a funnel?**
Running a full DNN ranker over millions of items would take seconds; users expect results in hundreds of milliseconds. The funnel uses cheap approximate methods (ANN, LSH) to narrow candidates quickly, then applies expensive precise models only to the reduced set.

**Q: What is approximate nearest neighbor (ANN) and why use it over exact KNN?**
ANN finds items approximately close to a query vector, trading a small recall loss for orders-of-magnitude speedup. HNSW (hierarchical navigable small world graphs) and IVF (inverted file index) allow sub-millisecond retrieval over millions of vectors, which exact KNN cannot achieve.

**Q: How would you handle a model server outage during peak traffic?**
Circuit breaker: detect high error rate → immediately stop calling the model server → fall back to the next tier (GBDT/popularity ranker). This prevents cascading failures from timeouts piling up. Log all fallback events, page on-call, and have a rollback procedure ready.

**Q: What is the difference between online and batch prediction?**
Online prediction runs at request time, using real-time features and returning results within the latency SLA. Batch prediction runs offline on a schedule, stores results in a lookup table, and returns them at request time with very low latency — but results are stale. Batch is cheaper and works when freshness isn't critical.

## Key Takeaways
- Funnel: millions → thousands → hundreds → tens, with explicit latency budgets at each stage
- ANN (HNSW, IVF) enables sub-millisecond retrieval over millions of vectors vs exact KNN
- Prediction modes: batch (hours, cheap) / online (real-time) / pre-computed (fast, stale)
- Cache item embeddings (days), user embeddings (minutes), never fraud scores
- Model-server vs model-in-service: latency vs operational flexibility tradeoff
- Always design fallback: full model → lightweight model → popularity → static list